In [ ]:
import sys
import vega
import os
from pathlib import Path
import scanpy as sc
import pandas as pd
import numpy as np
import ast
from joblib import Parallel, delayed

REPO_ROOT = Path("../../../../..").resolve()
VEGA_REPRO = REPO_ROOT / "models_reproductibility" / "vega-reproducibility"
sys.path.append(str(VEGA_REPRO / "src"))
import vanilla_vae
import train_vanilla_vae_suppFig1
import utils
from utils import *
from learning_utils import *
from vanilla_vae import VanillaVAE
from vega_model import VEGA

sys.path.append(str(REPO_ROOT / "utils" / "utils_evaluation"))
import utils_train_models
from utils_train_models import *
import vega_training
from vega_training import *
import vega_utils
from vega_utils import *
import utils_classification_emb
from utils_classification_emb import *

sys.path.append(str(REPO_ROOT / "utils" / "utils_interpretability"))
import utils_vega_perturbation
from utils_vega_perturbation import *


In [ ]:
device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')


In [ ]:
dir_path = str(REPO_ROOT / "notebooks" / "review VAEs" / "other datasets")


# Get data

In [ ]:
pathway_file = str(VEGA_REPRO / "data" / "reactomes.gmt")
path_data = str(REPO_ROOT / "datasets" / "tischdb") + "/"


In [ ]:
name_dataset = "CHOL_GSE138709"
type_data = 'tisch'


In [ ]:
name_model = 'Vega'
select_hvg = True
n_top_genes = 2000
random_seed = 42
train_size = 0.9
preprocess = True


In [ ]:
#Load data
if type_data == 'tisch':
    metadata = pd.read_csv(path_data + f'{name_dataset}_CellMetainfo_table.tsv', sep="\t")
    adata = sc.read_10x_h5(path_data + f'{name_dataset}_expression.h5', genome='GRCh38', gex_only=False)
    adata.obs = metadata
    column_labels_name = 'Celltype (major-lineage)' 
if name_dataset == 'Kang_PBMC':
    adata = sc.read(str(REPO_ROOT / "datasets" / "VEGA" / "Kang PBMC" / "train_pbmc.h5ad"))
    column_labels_name = 'cell_type' 

#prepare data
adata, pathway_dict, pathway_mask, list_pathways, df_genespathways, overlap_matrix = access_data_vega(None, adata, pathway_file)
adata, adata_train, adata_test, train_data, test_data, pathway_dict, pathway_mask = create_vega_training_data(name_model, preprocess, select_hvg, n_top_genes, random_seed, train_size, column_labels_name, adata, pathway_file)
adata, adata_train, adata_val, train_data, val_data, pathway_dict, pathway_mask = create_vega_training_data(name_model, False, select_hvg, n_top_genes, random_seed, train_size, column_labels_name, adata_train, pathway_file)


# Load model

In [ ]:
dir_path = str(REPO_ROOT / "notebooks" / "review VAEs" / "other datasets") + "/"
with open(dir_path+'parameters.txt', "r") as f:
    data_str = f.read() 
    dict_parameters = ast.literal_eval(data_str)
params = dict_parameters[name_model][name_dataset]
n_epochs = params['n_epochs']
lr = params['lr']
batch_size = params['batch_size']
beta = params['beta']
dropout = params['dropout']
val_resolution = params['val_resolution']
train_p = params['train_p']
test_p = params['test_p']
dev = params['dev']
save_path = params['save_path']
pos_dec = params['pos_dec']


In [ ]:
path_model_saved = dir_path + 'vega/interpretability evaluation/model/'


In [ ]:
model = VEGA(pathway_mask, positive_decoder=True, device=torch.device("cpu"))
state_dict = torch.load(path_model_saved + f"trained_vega_original_{name_dataset}.pt", map_location=torch.device("cpu"))
model.load_state_dict(state_dict)
model.eval()


# Apply perturbations

In [ ]:
path_to_save = f'/home/data/Vega/{name_dataset}/perturbations'
path_to_save_reconstructed = path_to_save + '/reconstructed'
path_to_save_embeddings = path_to_save + '/embeddings'

os.makedirs(path_to_save, exist_ok=True)
os.makedirs(path_to_save_reconstructed, exist_ok=True)
os.makedirs(path_to_save_embeddings, exist_ok=True)


In [ ]:
perturbation = 'inhibition'
mode = 'one_vs_all'
list_pathways = list(pathway_dict.keys())+['UNANNOTATED_'+str(k) for k in range(1)]


In [ ]:
split = 'test'


In [ ]:
results = Parallel(n_jobs=-1)(
    delayed(lambda p: vega_perturbation(
        model, name_dataset, adata_test.copy(), p,
        list_pathways, pathway_dict, perturbation,
        mode, split, path_to_save_embeddings,
        path_to_save_reconstructed
    ))(pathway_selected)
    for pathway_selected in list_pathways[:-1]
)
